# 02 - Elasticity (interactive companion)

Companion to `scripts/run_elasticity.py` and `reports/03_ELASTICITY_ANALYSIS.md`.

> Nothing here is causal: prices were set by the retailer, not randomised.


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pricing_engine.config import load_config
cfg = load_config(ROOT / "configs" / "config.yaml")
pd.set_option("display.width", 140)


In [ ]:
df = pd.read_parquet(cfg.path("features_table"),
                     columns=["upc", "store", "week", "move", "effective_unit_price",
                              "recorded_promotion_flag", "week_of_year", "time_index",
                              "descrip"])
df["sin52"] = np.sin(2 * np.pi * df["week_of_year"] / 52.0)
df["cos52"] = np.cos(2 * np.pi * df["week_of_year"] / 52.0)
df["trend"] = df["time_index"] / 100.0
len(df)

## 1. Descriptive arc elasticity (consecutive weeks)

In [ ]:
from pricing_engine.economics.elasticity import pairwise_arc_elasticities

arc = pairwise_arc_elasticities(df, ["upc", "store"])
e = arc["arc_elasticity"].dropna()
print(f"{len(e):,} usable pairs of {len(arc):,} "
      f"({100*(1-len(e)/len(arc)):.1f}% undefined, mostly because price did not change)")
print("median:", round(float(e.median()), 3))

## 2. Naive vs controlled log-log

The point of the phase: watch the coefficient move as controls are added.

In [ ]:
from pricing_engine.economics.elasticity import fit_loglog

fits = [
    fit_loglog(df, name="M1 naive pooled", sample_size=300_000),
    fit_loglog(df, name="M2 + UPC FE", absorb=["upc"], sample_size=300_000),
    fit_loglog(df, name="M3 + UPC x store FE", absorb=["upc", "store"], sample_size=300_000),
    fit_loglog(df, name="M4 + promo/season/trend", absorb=["upc", "store"],
               controls=["recorded_promotion_flag", "sin52", "cos52", "trend"],
               sample_size=300_000),
]
pd.DataFrame([f.as_dict() for f in fits])[
    ["model", "price_elasticity", "std_error", "r_squared", "n_obs"]
]

## 3. Heterogeneity across products

In [ ]:
from pricing_engine.economics.elasticity import elasticity_by_group

by_upc = elasticity_by_group(df, ["upc"], min_obs=200, min_distinct_prices=10)
print(f"{len(by_upc)} UPCs estimated | median {by_upc['elasticity'].median():.2f} | "
      f"{100*(by_upc['elasticity'] < -1).mean():.0f}% elastic | "
      f"{100*((by_upc['elasticity'] > 0) & (by_upc['t_stat'] > 1.96)).mean():.1f}% wrong-signed significant")
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(by_upc["elasticity"].clip(-8, 4), bins=50)
ax.axvline(-1, color="crimson", ls="--")
plt.show()

## 4. The promotion confound

In [ ]:
for label, sub in (("recorded promo", df[df.recorded_promotion_flag == 1]),
                   ("no recorded promo", df[df.recorded_promotion_flag == 0])):
    f = fit_loglog(sub, name=label, absorb=["upc", "store"], sample_size=200_000)
    print(f"{label:20s} elasticity {f.price_elasticity:+.3f}  (n={f.n_obs:,})")

## 5. What the trained model itself implies

The number that actually drives recommendations is the model's own implied
elasticity, measured by finite difference on the fitted model.

In [ ]:
from pricing_engine.models.demand_model import load_model
from pricing_engine.features.build import training_frame

model = load_model(cfg.path("models_dir") / "demand_model.joblib", cfg=cfg)
feats = training_frame(pd.read_parquet(cfg.path("features_table")))
test = feats[feats["week"] >= model.metadata.test_weeks[0]]
model.implied_elasticity(test, sample=50_000)